# 02 · Retrieval Analysis
Explores the FAISS retriever built in Phase 3 (`src/embeddings.py`, `src/retriever.py`) across all three supported languages.

In [ ]:
import sys
from pathlib import Path

# Notebooks run from notebooks/, but the src package lives at the project
# root -- add the project root to sys.path so `from src import ...` works.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

pd.set_option("display.max_colwidth", 100)


In [ ]:
from src.retriever import Retriever

retriever = Retriever()
retriever.build_index()
print(f'Index size: {retriever.index.ntotal} documents')

## Sample multilingual retrieval
The same underlying question (capital of India) asked in all three languages, restricted to same-language retrieval.

In [ ]:
samples = [
    ('What is the capital of India?', 'English'),
    ('भारत की राजधानी क्या है?', 'Hindi'),
    ('ભારતની રાજધાની શું છે?', 'Gujarati'),
]

for question, language in samples:
    print(f'--- {language}: {question} ---')
    results = retriever.retrieve(question, language, top_k=3)
    for rank, doc in enumerate(results, start=1):
        print(f'  Rank {rank} | score={doc.similarity_score:.4f} | {doc.document_id} | {doc.content[:70]}...')
    print()

## Retrieval quality check (all 45 questions)
Loaded from `outputs/tables/retrieval_quality_details.csv` and `retrieval_quality_summary.csv`, generated by `src/evaluate_retrieval.py` (Phase 3). Top-1 accuracy compares the top retrieved document's linked fact against the fact the question was written about.

In [ ]:
from src import config

details_df = pd.read_csv(config.RETRIEVAL_QUALITY_DETAILS_CSV, encoding='utf-8-sig')
summary_df = pd.read_csv(config.RETRIEVAL_QUALITY_SUMMARY_CSV, encoding='utf-8-sig')
summary_df

In [ ]:
incorrect = details_df[~details_df['top1_correct']]
print(f'Questions with incorrect top-1 retrieval: {len(incorrect)}')
incorrect[['question_id', 'language', 'category', 'expected_fact_id', 'retrieved_fact_id']]

## Similarity score distribution by language

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
languages = config.SUPPORTED_LANGUAGES
data = [details_df[details_df.language == lang]['similarity_score'].dropna() for lang in languages]
ax.boxplot(data, tick_labels=languages, patch_artist=True)
ax.set_title('Top-1 Retrieval Similarity Score by Language')
ax.set_ylabel('Cosine Similarity')
plt.show()

## Notes
- Retrieval uses `intfloat/multilingual-e5-base` embeddings with `query: ` / `passage: ` prefixes, L2-normalized, searched via FAISS `IndexFlatIP` (exact inner-product search, equivalent to cosine similarity on normalized vectors).
- `IndexFlatIP` is appropriate at this scale (54 documents) -- an approximate index (IVF/HNSW) would add complexity with no practical benefit here.
- Any retrieval errors reflect real embedding-model behavior on this dataset, not implementation bugs -- worth documenting honestly rather than hiding.